# Generate admission-level predictions

Loads fitted models and creates one common prediction table for downstream evaluation, temporal analysis, subgroup analysis, and explainability.

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
from pathlib import Path
import joblib
import pandas as pd
import yaml
from icare_risk.data.audit import load_admission_file
from icare_risk.models.prediction import generate_prediction_table

CONFIG_PATH = Path('../../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

df = load_admission_file(config['outputs']['processed_dataset'])
model_directory = Path(config['outputs']['model_directory'])
fitted_models = {}
feature_map = {}
for model_cfg in config['models']:
    name = model_cfg['name']
    bundle = joblib.load(model_directory / f'{name}.joblib')
    fitted_models[name] = bundle['model']
    feature_map[name] = bundle['features']

prediction_table = generate_prediction_table(
    df, fitted_models, feature_map,
    id_columns=[config['columns']['admission_id'], config['columns']['patient_id']],
    target_column=config['columns']['target'],
    sort_by_model=None,  # e.g. 'xgb_all_scores' to sort highest risk first
)
display(prediction_table.head())

,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,GENDER_DESC,UPDATE_DT_TM,esbl_label_test,split,lr_kim_components_prob,lr_all_score_values_prob,lr_clinical_baseline_prob,xgb_kim_components_prob,xgb_all_score_values_prob,xgb_clinical_baseline_prob
0,10001,7929089,2005940,4725136,2020-06-10,22:22:00,35,2020-07-03,320,cardiology,...,Unknown,2023-03-28,0,train,0.200105,0.199277,0.193409,0.199943,0.204228,0.239159
1,10002,8200425,6570033,6896867,2020-03-27,17:31:00,63,2020-03-28,301,gastroenterology,...,Unknown,2023-03-26,0,train,0.200105,0.199957,0.190516,0.199943,0.196213,0.153451
2,10003,9931252,5492977,1426665,2023-01-06,00:18:00,89,2023-01-25,301,gastroenterology,...,Indeterminate,2023-04-04,0,test,0.200105,0.201321,0.121112,0.199943,0.197526,0.074811
3,10004,5095152,3730006,3105590,2022-09-14,10:32:00,25,2022-10-12,301,gastroenterology,...,Unknown,2023-05-10,0,validation,0.200105,0.199277,0.194451,0.199943,0.204228,0.183218
4,10005,7807609,7413177,1104787,2023-09-08,10:55:00,80,2023-09-23,140,oral surgery,...,Male,2023-07-14,1,test,0.200105,0.200979,0.200254,0.199943,0.211959,0.198743


In [6]:
output_path = Path('results/predictions/admission_predictions.parquet')
output_path.parent.mkdir(parents=True, exist_ok=True)
prediction_table.to_parquet(output_path, index=False)
print(f'Saved {len(prediction_table):,} admission predictions to {output_path}')

Saved 10,000 admission predictions to results/predictions/admission_predictions.parquet
